# HAKE-MER — RoBERTa full ladder (M1 → M4)

**PFE — Mariem El Wedani** · SIIVA · Encadrant académique : Sahbi Bahroun · 2025–2026


Single Colab session: `./run_roberta_full_ablation.sh` (+M1, +M1+M2, +M3 NRC/SenticNet, +M4 SenticNet).

**GPU:** A100 or T4 recommended (9 training runs × 3 seeds).

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

import os
REPO = os.environ.get("GITHUB_REPO", "khalef-khalil/marii")
WORKDIR = Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
!chmod +x run_roberta_full_ablation.sh run_roberta_ablation.sh run_roberta_m3_m4_ablation.sh
!./run_roberta_full_ablation.sh {TRAIN_FLAGS}

Zip all RoBERTa campaign JSON from `reference/artifacts/` matching `*roberta_base_campaign.json` and download. Integrate locally with `integrate_roberta_m3_m4_artifacts.py` (M3/M4 zip) plus existing M1/M2 workflow.